# Error Analysis: LLM-as-Judge Evaluation

Manual calibration of the evaluation pipeline. We investigate:
1. Which questions fail systematically and why
2. Whether the TF-IDF retriever finds the right statements
3. Whether the LLM judge scores consistently
4. Root causes behind low scores

In [1]:
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

EVAL_CSV = Path("../results/evaluated_results_v1.csv")
STATEMENTS_JSON = Path("../data/statements/statements_2026_diverse.json")

with open(EVAL_CSV, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(f"Loaded {len(rows)} evaluated answers")
print(f"Columns: {list(rows[0].keys())}")

Loaded 800 evaluated answers
Columns: ['model', 'context', 'question_id', 'difficulty', 'category', 'question_en', 'reference_answer', 'model_answer', 'token_budget', 'context_chars', 'judge_score', 'judge_reasoning']


## 1. Score distribution overview

In [2]:
# Overall score distribution
score_counts = Counter(int(r["judge_score"]) for r in rows)
total = len(rows)

print("Score distribution:")
for score in sorted(score_counts):
    count = score_counts[score]
    bar = "█" * (count // 10)
    print(f"  {score:>2}: {count:>4} ({count/total*100:.1f}%)  {bar}")

# By context type
print("\nAverage score by context type:")
by_ctx_type = defaultdict(list)
for r in rows:
    ctx_type = r["context"].split("_")[0]  # "statements" or "markdown"
    by_ctx_type[ctx_type].append(int(r["judge_score"]))

for ctx, scores in sorted(by_ctx_type.items()):
    valid = [s for s in scores if s >= 0]
    print(f"  {ctx}: {sum(valid)/len(valid):.2f} (n={len(valid)})")

Score distribution:
  -1:    6 (0.8%)  
   0:  459 (57.4%)  █████████████████████████████████████████████
   1:   29 (3.6%)  ██
   2:  136 (17.0%)  █████████████
   3:  170 (21.2%)  █████████████████

Average score by context type:
  markdown: 0.92 (n=399)
  statements: 1.12 (n=395)


## 2. Which questions fail most often?

If a question scores 0 across most models and contexts, the problem is systemic — either the question is too hard, the reference answer is wrong, or the context never contains the needed information.

In [3]:
q_zeros = Counter()
q_total = Counter()
q_text = {}
q_ref = {}
q_diff = {}

for r in rows:
    qid = r["question_id"]
    q_total[qid] += 1
    q_text[qid] = r["question_en"]
    q_ref[qid] = r["reference_answer"]
    q_diff[qid] = r["difficulty"]
    if int(r["judge_score"]) == 0:
        q_zeros[qid] += 1

print(f"{'QID':>4} {'Diff':<7} {'Zeros':>7} {'Total':>6} {'Rate':>6}  Question")
print("-" * 90)
for qid, cnt in sorted(q_zeros.items(), key=lambda x: -x[1]):
    total = q_total[qid]
    rate = cnt / total * 100
    print(f"Q{qid:>3} {q_diff[qid]:<7} {cnt:>5}/{total:<4} {rate:>5.0f}%  {q_text[qid][:75]}")

 QID Diff      Zeros  Total   Rate  Question
------------------------------------------------------------------------------------------
Q 18 hard       37/40      92%  Which program under the Ministry of Emergency Situations receives the most 
Q 16 hard       36/40      90%  What share of the total 2026 program budget goes to the Ministry of Finance
Q  7 medium     34/40      85%  What is the total budget across all program-level items for 2026?
Q 19 hard       33/40      82%  How many total budget programs and budget measures are in the 2026 program-
Q  1 easy       30/40      75%  What is the 2026 funding for the 'School Education' program under the Minis
Q  9 medium     29/40      72%  Does the total program-level budget grow or shrink between 2026 and 2027, a
Q 17 hard       27/40      68%  Which budget measure (not program) has the largest 2026 funding?
Q 12 medium     25/40      62%  What is the total 2026 program funding for the Ministry of Transport and Co
Q 15 hard       25/40

## 3. Statements vs markdown failure modes

For the worst questions, check: does the failure happen in both context modes, or only one?

In [12]:
# For each question, break down zeros by context type
worst_questions = [qid for qid, _ in sorted(q_zeros.items(), key=lambda x: -x[1])[:10]]

print(f"{'QID':>4}  {'stmt_zeros':>11} {'md_zeros':>9}  {'stmt_avg':>9} {'md_avg':>7}  Question")
print("-" * 90)

for qid in worst_questions:
    stmt_rows = [r for r in rows if r["question_id"] == qid and "statements" in r["context"]]
    md_rows = [r for r in rows if r["question_id"] == qid and "markdown" in r["context"]]
    
    stmt_zeros = sum(1 for r in stmt_rows if int(r["judge_score"]) == 0)
    md_zeros = sum(1 for r in md_rows if int(r["judge_score"]) == 0)
    
    stmt_scores = [int(r["judge_score"]) for r in stmt_rows if int(r["judge_score"]) >= 0]
    md_scores = [int(r["judge_score"]) for r in md_rows if int(r["judge_score"]) >= 0]
    
    stmt_avg = sum(stmt_scores) / len(stmt_scores) if stmt_scores else 0
    md_avg = sum(md_scores) / len(md_scores) if md_scores else 0
    
    print(f"Q{qid:>3}  {stmt_zeros:>5}/{len(stmt_rows):<5} {md_zeros:>4}/{len(md_rows):<4}  {stmt_avg:>8.2f} {md_avg:>7.2f}  {q_text[qid][:55]}")

 QID   stmt_zeros  md_zeros   stmt_avg  md_avg  Question
------------------------------------------------------------------------------------------
Q 18     20/20      17/20        0.00    0.45  Which program under the Ministry of Emergency Situation
Q 16     19/20      17/20        0.05    0.25  What share of the total 2026 program budget goes to the
Q  7     20/20      14/20        0.00    0.60  What is the total budget across all program-level items
Q 19     20/20      13/20        0.00    0.40  How many total budget programs and budget measures are 
Q  1     20/20      10/20        0.00    1.50  What is the 2026 funding for the 'School Education' pro
Q  9     16/20      13/20        0.40    0.70  Does the total program-level budget grow or shrink betw
Q 17     11/20      16/20        1.30    0.60  Which budget measure (not program) has the largest 2026
Q 12      8/20      17/20        1.65    0.30  What is the total 2026 program funding for the Ministry
Q 15     11/20      14/20   

## 4. Retrieval diagnosis

The core question: when statements mode fails, is it because the **retriever didn't find** the relevant statement, or because the **model couldn't use** it?

We rebuild the TF-IDF retriever and check where the target statement ranks for each failing question.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Load statements
with open(STATEMENTS_JSON, encoding="utf-8") as f:
    stmt_data = json.load(f)

all_statements = []
for row in stmt_data:
    for s in row.get("statements_en", []):
        all_statements.append(s)
    for s in row.get("statements_ru", []):
        all_statements.append(s)

print(f"Total statements: {len(all_statements)}")

# Build retriever (same config as run_experiments.py)
vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5),
    max_features=50000, sublinear_tf=True
)
matrix = vectorizer.fit_transform(all_statements)
print(f"TF-IDF matrix: {matrix.shape}")

Total statements: 6658
TF-IDF matrix: (6658, 50000)


In [6]:
def retrieve_and_rank(question, top_k=10):
    """Return top-k statements and their scores for a given question."""
    q_vec = vectorizer.transform([question])
    scores = (matrix @ q_vec.T).toarray().squeeze()
    top_idx = scores.argsort()[-top_k:][::-1]
    results = []
    for i in top_idx:
        results.append((scores[i], all_statements[i]))
    return results, scores


def find_target_rank(scores, keyword):
    """Find the rank of the first statement containing the keyword."""
    sorted_idx = scores.argsort()[::-1]
    for rank, i in enumerate(sorted_idx):
        if keyword.lower() in all_statements[i].lower():
            return rank + 1, scores[i], all_statements[i]
    return None, None, None

In [7]:
# Test retrieval for the worst-performing questions
# Each entry: (question_id, question_text, keyword to search for in target statement)
test_cases = [
    ("1", "What is the 2026 funding for the 'School Education' program under the Ministry of Education?",
     "Школьное образование"),
    ("18", "Which program under the Ministry of Emergency Situations receives the most 2026 funding?",
     "чрезвычайных ситуаций"),
    ("7", "What is the total budget across all program-level items for 2026?",
     "593"),
    ("12", "What is the total 2026 program funding for the Ministry of Transport and Communications?",
     "транспорт"),
    ("2", "How many budget programs does the Ministry of Health have?",
     "здравоохранения"),
]

print(f"{'QID':>4}  {'Target rank':>12}  {'Score':>6}  Question → Target keyword")
print("=" * 90)

for qid, question, keyword in test_cases:
    top_results, scores = retrieve_and_rank(question)
    rank, target_score, target_stmt = find_target_rank(scores, keyword)
    
    rank_str = f"#{rank}" if rank else "NOT FOUND"
    score_str = f"{target_score:.4f}" if target_score else "—"
    
    print(f"\nQ{qid:>3}  {rank_str:>12}  {score_str:>6}  \"{question[:60]}\"")
    print(f"       keyword: \"{keyword}\"")
    if rank and rank > 30:
        print(f"       ⚠ RETRIEVAL FAILURE — rank {rank} is outside top-30 cutoff")
    elif rank and rank <= 30:
        print(f"       ✓ Retrieved successfully (rank {rank})")
    
    # Show top 3 retrieved (what the model actually sees)
    print(f"       Top 3 actually retrieved:")
    for i, (sc, stmt) in enumerate(top_results[:3]):
        print(f"         {i+1}. [{sc:.4f}] {stmt[:100]}")

 QID   Target rank   Score  Question → Target keyword

Q  1          #213  0.1698  "What is the 2026 funding for the 'School Education' program "
       keyword: "Школьное образование"
       ⚠ RETRIEVAL FAILURE — rank 213 is outside top-30 cutoff
       Top 3 actually retrieved:
         1. [0.2626] The total 2026 allocation for 'Министерство финансов Кыргызской Республики' is 15.5 billion som.
         2. [0.2591] The total 2026 allocation for 'Министерство юстиции Кыргызской Республики' is 1.4 billion som.
         3. [0.2492] The total 2026 allocation for 'Министерство иностранных дел Кыргызской Республики' is 4.7 billion so

Q 18           #16  0.1936  "Which program under the Ministry of Emergency Situations rec"
       keyword: "чрезвычайных ситуаций"
       ✓ Retrieved successfully (rank 16)
       Top 3 actually retrieved:
         1. [0.2150] Ministry 'Министерство финансов Кыргызской Республики' (code 25) is allocated budget programs in the
         2. [0.2101] Ministry 'Мин

## 5. Cross-lingual retrieval gap

The hypothesis: English queries can't match Russian entity names via character n-grams because Latin and Cyrillic share zero characters. Let's measure this directly.

In [13]:
# Load QA pairs to get all questions
with open("../data/qa/qa_pairs_2026.json", encoding="utf-8") as f:
    qa_pairs = json.load(f)["qa_pairs"]

print(f"{'QID':>4} {'Diff':<7} {'Top1':>6} {'Top30avg':>9} {'HasCyr':>7}  Question")
print("-" * 90)

for qa in qa_pairs:
    qid = str(qa["id"])
    question = qa.get("question_en", qa["question"])
    
    top_results, scores = retrieve_and_rank(question, top_k=30)
    top1_score = top_results[0][0] if top_results else 0
    top30_avg = np.mean([s for s, _ in top_results]) if top_results else 0
    
    # Does the question contain any Cyrillic characters?
    has_cyrillic = any("\u0400" <= c <= "\u04FF" for c in question)
    
    # Get average judge score for this question in statements mode
    stmt_scores = [int(r["judge_score"]) for r in rows 
                   if r["question_id"] == qid and "statements" in r["context"] 
                   and int(r["judge_score"]) >= 0]
    stmt_avg = sum(stmt_scores) / len(stmt_scores) if stmt_scores else 0
    
    flag = "⚠" if top1_score < 0.2 else " "
    print(f"Q{qid:>3} {qa['difficulty']:<7} {top1_score:>6.3f} {top30_avg:>9.3f} {str(has_cyrillic):>7} {flag} {question[:55]}")

 QID Diff      Top1  Top30avg  HasCyr  Question
------------------------------------------------------------------------------------------
Q  1 easy     0.263     0.237   False   What is the 2026 funding for the 'School Education' pro
Q  2 easy     0.505     0.457   False   How many budget programs does the Ministry of Health ha
Q  3 easy     0.330     0.299   False   Which program has the largest funding allocation in the
Q  4 easy     0.322     0.291   False   How many total ministries and agencies are represented 
Q  5 easy     0.295     0.257   False   What is the total 2026 funding for the Jogorku Kenesh (
Q  6 medium   0.332     0.302   False   What is the total 2026 funding across all programs of t
Q  7 medium   0.313     0.290   False   What is the total budget across all program-level items
Q  8 medium   0.270     0.245   False   Which ministry receives more funding: Ministry of Healt
Q  9 medium   0.465     0.367   False   Does the total program-level budget grow or shrink be

## 6. Judge consistency check

For the same question at the same token budget, different models may get different scores — that's expected. But if the **same answer content** gets wildly different scores, the judge is inconsistent.

In [9]:
# Find cases where same (question, budget) has both score=0 and score=3
by_q_budget = defaultdict(list)
for r in rows:
    by_q_budget[(r["question_id"], r["token_budget"])].append(r)

conflicts = []
for (qid, bud), group in by_q_budget.items():
    scores = [int(r["judge_score"]) for r in group]
    if 0 in scores and 3 in scores:
        conflicts.append((qid, bud, group))

print(f"Found {len(conflicts)} (question, budget) pairs with both score=0 and score=3\n")

# Examine a few
for qid, bud, group in conflicts[:5]:
    print(f"Q{qid} at {bud} tokens")
    print(f"  Reference: {group[0]['reference_answer'][:80]}")
    for r in sorted(group, key=lambda x: -int(x["judge_score"])):
        score = int(r["judge_score"])
        ans = r["model_answer"][:90].replace("\n", " ")
        ctx = r["context"]
        # Flag suspicious: model refuses but gets score > 0
        is_refusal = any(phrase in r["model_answer"].lower()[:150] 
                        for phrase in ["does not contain", "cannot", "no information", 
                                       "не содержит", "insufficient"])
        flag = " ⚠ REFUSAL WITH HIGH SCORE" if is_refusal and score >= 2 else ""
        print(f"  score={score}  {r['model']:20s} {ctx:22s}  {ans}{flag}")
    print()

Found 26 (question, budget) pairs with both score=0 and score=3

Q6 at 2000 tokens
  Reference: 56,424,894.11 тыс. сом
  score=3  gemma4               statements_2000tok      The provided context does not contain information regarding a "Ministry of Education"; how ⚠ REFUSAL WITH HIGH SCORE
  score=3  deepseek-v3.2        statements_2000tok      56.4 billion som
  score=3  deepseek-v3.2-thinking statements_2000tok      The context does not contain information about a "Министерство образования" (Ministry of E ⚠ REFUSAL WITH HIGH SCORE
  score=3  mistral-small-4      statements_2000tok      56.4 billion som.
  score=3  qwen3.5              statements_2000tok      56.4 billion som for 'Министерство просвещения Кыргызской Республики'.
  score=0  gemma4               markdown_2000tok        The provided context does not contain information regarding the Ministry of Education.
  score=0  deepseek-v3.2        markdown_2000tok        The provided context does not contain any entries for the Mi

In [10]:
# Count judge errors: refusal answers that scored >= 2
refusal_phrases = ["does not contain", "cannot answer", "no information", 
                   "не содержит", "insufficient", "unable to", "not available",
                   "don't have", "context doesn't"]

suspect_scores = []
for r in rows:
    score = int(r["judge_score"])
    answer_lower = r["model_answer"].lower()[:200]
    is_refusal = any(phrase in answer_lower for phrase in refusal_phrases)
    if is_refusal and score >= 2:
        suspect_scores.append(r)

print(f"Suspect judge scores (refusal answer but score >= 2): {len(suspect_scores)} out of {len(rows)}")
print(f"That's {len(suspect_scores)/len(rows)*100:.1f}% of all evaluations\n")

if suspect_scores:
    print("Examples:")
    for r in suspect_scores[:5]:
        print(f"  Q{r['question_id']} score={r['judge_score']} | {r['model']:20s} {r['context']}")
        print(f"    Answer: {r['model_answer'][:120]}")
        print(f"    Judge:  {r['judge_reasoning'][:120]}")
        print()

Suspect judge scores (refusal answer but score >= 2): 5 out of 800
That's 0.6% of all evaluations

Examples:
  Q6 score=3 | gemma4               statements_2000tok
    Answer: The provided context does not contain information regarding a "Ministry of Education"; however, the total 2026 allocatio
    Judge:  Model provides the correct total funding value (≈56.4 billion som) matching the reference amount.

  Q6 score=2 | gemma4               markdown_16000tok
    Answer: The provided context does not contain a "Ministry of Education"; however, it lists the "Министерство просвещения Кыргызс
    Judge:  Identifies the correct ministry but provides an incorrect total funding amount.

  Q6 score=2 | gemma4               markdown_full
    Answer: The provided context does not contain a "Ministry of Education"; however, it lists the "Министерство просвещения Кыргызс
    Judge:  Correct entity (Ministry of Education) but the total funding figure is incorrect compared to the reference value.

  

## 7. Summary of findings

Categorize each question's failure mode.

In [11]:
# Classify failure modes for each question
print("FAILURE MODE CLASSIFICATION")
print("=" * 80)

for qa in qa_pairs:
    qid = str(qa["id"])
    question = qa.get("question_en", qa["question"])
    
    # Scores by context type
    stmt_scores = [int(r["judge_score"]) for r in rows 
                   if r["question_id"] == qid and "statements" in r["context"]
                   and int(r["judge_score"]) >= 0]
    md_scores = [int(r["judge_score"]) for r in rows 
                 if r["question_id"] == qid and "markdown" in r["context"]
                 and int(r["judge_score"]) >= 0]
    
    stmt_avg = sum(stmt_scores) / len(stmt_scores) if stmt_scores else 0
    md_avg = sum(md_scores) / len(md_scores) if md_scores else 0
    
    # Classify
    if stmt_avg < 0.5 and md_avg < 0.5:
        mode = "BOTH FAIL — question may require aggregation or cross-row reasoning"
    elif stmt_avg < 0.5 and md_avg >= 1.0:
        mode = "RETRIEVAL FAILURE — info exists but retriever can't find it"
    elif stmt_avg >= 1.0 and md_avg < 0.5:
        mode = "MARKDOWN PARSING FAILURE — model can't extract from raw table"
    elif stmt_avg >= 1.5 and md_avg >= 1.5:
        mode = "BOTH OK"
    else:
        mode = f"MIXED (stmt={stmt_avg:.1f}, md={md_avg:.1f})"
    
    zero_rate = q_zeros.get(qid, 0) / q_total[qid] * 100
    print(f"\nQ{qid:>2} [{qa['difficulty']:6s}] {zero_rate:>4.0f}% zeros | stmt={stmt_avg:.2f} md={md_avg:.2f}")
    print(f"    {question[:70]}")
    print(f"    → {mode}")

FAILURE MODE CLASSIFICATION

Q 1 [easy  ]   75% zeros | stmt=0.00 md=1.50
    What is the 2026 funding for the 'School Education' program under the 
    → RETRIEVAL FAILURE — info exists but retriever can't find it

Q 2 [easy  ]   45% zeros | stmt=1.16 md=1.00
    How many budget programs does the Ministry of Health have?
    → MIXED (stmt=1.2, md=1.0)

Q 3 [easy  ]   52% zeros | stmt=0.50 md=2.10
    Which program has the largest funding allocation in the 2026 budget?
    → MIXED (stmt=0.5, md=2.1)

Q 4 [easy  ]   50% zeros | stmt=1.25 md=0.75
    How many total ministries and agencies are represented in the program-
    → MIXED (stmt=1.2, md=0.8)

Q 5 [easy  ]    2% zeros | stmt=3.00 md=2.79
    What is the total 2026 funding for the Jogorku Kenesh (parliament)?
    → BOTH OK

Q 6 [medium]   25% zeros | stmt=3.00 md=1.15
    What is the total 2026 funding across all programs of the Ministry of 
    → MIXED (stmt=3.0, md=1.1)

Q 7 [medium]   85% zeros | stmt=0.00 md=0.60
    What is t